In [ ]:
# Run this first to clone the published DSA implementation.
from pathlib import Path
import subprocess
import sys
import os
import json
import hashlib
import shutil

REPO_URL = "https://github.com/sttruji/dynamic-state-anchoring.git"
REPO_REF = "main"  # Set a commit SHA here to reproduce a particular implementation.
REPO_DIR = Path("/content/dynamic-state-anchoring")

def git_output(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()

if not REPO_DIR.exists():
    try:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("Check that the repository is reachable, then rerun. "
                           "Private repositories need authenticated Git access configured separately.") from exc
else:
    remote = git_output("remote", "get-url", "origin").rstrip("/").removesuffix(".git")
    if remote != REPO_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError("Existing checkout has a different remote; choose another REPO_DIR.")
    if git_output("status", "--porcelain"):
        raise RuntimeError("Existing checkout has local changes. Preserve them before updating it.")
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
REPO_COMMIT = git_output("rev-parse", "HEAD")
if not (REPO_DIR / "requirements-colab.txt").is_file():
    raise RuntimeError("This ref does not contain the DSA implementation.")
print("DSA implementation:", REPO_COMMIT)


# Dynamic State Anchoring · Colab pilot

Choose **Runtime → Change runtime type → GPU**, then run cells in order. The first
cell clones `sttruji/dynamic-state-anchoring`. No credentials belong in this file.

This is an implementation pilot: 32 unchanged LOTUS training rows, 200 optimizer
updates per arm, and 32 validation questions by default. It is not a benchmark
claim. Static and learned arms use the same initialization, data order, backbone,
and losses. Enable all four write policies only after this small pilot works.
Data, Hugging Face downloads, checkpoints, and reports persist on your Drive.


In [ ]:
# Reuse Colab's CUDA/PyTorch stack; isolate project packages from the notebook kernel.
# The helper uses --without-pip --system-site-packages and targets this environment explicitly.
VENV_DIR = Path("/content/dsa-venv")
subprocess.run([sys.executable, str(REPO_DIR / "scripts/setup_colab.py"),
                "--venv-dir", str(VENV_DIR),
                "--requirements", str(REPO_DIR / "requirements-colab.txt")], check=True)
PYTHON = str(VENV_DIR / "bin/python")
print("Project Python:", PYTHON)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/dynamic-state-anchoring")
for name in ("corpus", "prepared", "runs", "reports", "cache/huggingface"):
    (PROJECT_DIR / name).mkdir(parents=True, exist_ok=True)
# Set before any subprocess imports Hugging Face packages.
os.environ["HF_HOME"] = str(PROJECT_DIR / "cache/huggingface")
os.environ["HF_HUB_CACHE"] = str(PROJECT_DIR / "cache/huggingface/hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.environ["HF_HUB_CACHE"]
os.environ["TRANSFORMERS_CACHE"] = os.environ["HF_HUB_CACHE"]
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
print("Persistent storage:", PROJECT_DIR)


In [ ]:
# Freeze these before training; changed experimental settings need a new name.
EXPERIMENT = "gpt2-pilot-v1"
BACKBONE = "yingfanbot/gsm-cot-gpt2"
MODEL_REVISION = "e0ef1887404adbf6c531d2cac7804694321f87bd"
DATA_REVISION = "e06a32ee5e4cd117171daeb4755d2a97ece62761"
SEED = 7
TRAIN_LIMIT = 32  # 0 selects the full unchanged training split, after profiling.
LOOPS = 16
WORKSPACE = 4
STEPS = 200  # Total target updates. Increase to extend an existing run.
LEARNING_RATE = 3e-5
SAVE_EVERY = 25
KEEP_CHECKPOINTS = 2
GRADIENT_ACCUMULATION = 1
ARMS = ["static", "learned"]  # Full comparison: static, every_loop, periodic, learned.
VALID_LIMIT = 32  # 0 evaluates all 500 validation questions.
EVAL_BUDGETS = [LOOPS]  # Predeclare a depth grid on validation, e.g. [4, 8, 16, 32].
MAX_NEW_TOKENS = 64
DEVICE = "cuda"

assert EXPERIMENT and Path(EXPERIMENT).name == EXPERIMENT
assert set(ARMS) <= {"static", "every_loop", "periodic", "learned"} and len(set(ARMS)) == len(ARMS)
assert ARMS and all(k > 0 for k in EVAL_BUDGETS)
PREP_NAME = f"{REPO_COMMIT[:12]}-train{TRAIN_LIMIT}-k{LOOPS}-s{SEED}"
PREPARED_DRIVE = PROJECT_DIR / "prepared" / PREP_NAME
PREPARED_LOCAL = Path("/content/dsa-prepared") / PREP_NAME
RUN_ROOT = PROJECT_DIR / "runs" / f"{EXPERIMENT}-{REPO_COMMIT[:12]}"
REPORT_ROOT = PROJECT_DIR / "reports" / f"{EXPERIMENT}-{REPO_COMMIT[:12]}"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
SMOKE_REPORT = REPORT_ROOT / "unmodified-checkpoint-smoke.json"

import runpy
_run_logged = runpy.run_path(str(REPO_DIR / "scripts/colab_process.py"))["run_logged"]

def run_project(*args):
    return _run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR,
                       env=os.environ.copy(), log_dir=REPORT_ROOT / "logs")

config = {key: globals()[key] for key in ("EXPERIMENT", "REPO_COMMIT", "BACKBONE", "MODEL_REVISION",
          "DATA_REVISION", "SEED", "TRAIN_LIMIT", "LOOPS", "WORKSPACE", "STEPS", "LEARNING_RATE",
          "ARMS", "VALID_LIMIT", "EVAL_BUDGETS", "MAX_NEW_TOKENS", "DEVICE")}
print(json.dumps(config, indent=2))


The next cell records the actual runtime, checks GPU access and runs the local
correctness suite. DSA dependencies are imported only by the isolated project
Python. A compatibility failure stops before downloading model weights or
starting training. Local development used CPU PyTorch 2.7.1; the hosted GPU
runtime still needs this check.


In [ ]:
runtime_code = r"""
import importlib.metadata, json, platform, sys
from pathlib import Path
import torch
from packaging.version import Version
from dsa.models.backbone import BackboneAdapter
assert (3, 10) <= sys.version_info[:2] < (3, 14), "Use a Python 3.10–3.13 runtime."
assert Version(torch.__version__.split('+')[0]) >= Version('2.7'), "Use PyTorch >=2.7."
requested_device, output = sys.argv[1:]
probe = {'requested_device': requested_device, 'python': platform.python_version(),
         'torch': torch.__version__, 'cuda_build': torch.version.cuda,
         'cuda_available': torch.cuda.is_available(), 'cuda_device_count': torch.cuda.device_count()}
print(json.dumps({'runtime_probe': probe}, indent=2), flush=True)
if requested_device == 'cuda' and not probe['cuda_available']:
    raise RuntimeError('CUDA is unavailable in this runtime. In Colab choose Runtime > Change runtime type > T4 GPU, '
                       'save/connect, then rerun setup. If a GPU is already allocated, inspect the runtime probe '
                       'and nvidia-smi output before changing packages.')
torch.set_num_threads(1)
model = BackboneAdapter.tiny(259).model.to(requested_device).eval()
with torch.no_grad():
    result = model(input_ids=torch.tensor([[3, 4, 5]], device=requested_device), use_cache=False)
assert bool(torch.isfinite(result.logits).all()), 'Nonfinite GPT-2 forward'
report = {'python': platform.python_version(), 'device': requested_device,
          'cuda_available': torch.cuda.is_available(), 'cuda': torch.version.cuda,
          'gpu': torch.cuda.get_device_name() if torch.cuda.is_available() else None,
          'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
          'packages': {name: importlib.metadata.version(name) for name in
                       ('torch', 'transformers', 'tokenizers', 'huggingface_hub', 'numpy')},
          'tiny_gpt2_forward_passed': True}
Path(output).write_text(json.dumps(report, indent=2) + '\n')
print(json.dumps(report, indent=2))
"""
run_project("-c", runtime_code, DEVICE, REPORT_ROOT / "runtime.json")
run_project("-m", "unittest", "discover", "-s", "tests", "-q")
print("Preflight passed. Runtime report saved on Drive.")


The corpus downloader checks pinned hashes and reuses verified Drive files on
later sessions. It downloads the original three splits once, then prepares a
deterministic training subset and retains every validation/test row in source
order. No-event and overflow training rows remain in the answer/explicit losses.
The data issues documented in the audit remain unchanged in this primary pilot.
Prepared inputs are copied to the VM for training; Drive remains the durable copy.


In [ ]:
run_project("scripts/prepare_dsa_pilot.py",
            "--storage-dir", PROJECT_DIR / "corpus", "--output-dir", PREPARED_DRIVE,
            "--train-limit", TRAIN_LIMIT, "--loops", LOOPS, "--seed", SEED,
            "--tokenizer-model", BACKBONE, "--revision", MODEL_REVISION)
shutil.copytree(PREPARED_DRIVE, PREPARED_LOCAL, dirs_exist_ok=True)
prepared_manifest = json.loads((PREPARED_LOCAL / "manifest.json").read_text())
print("Prepared records:", {name: entry["records"] for name, entry in prepared_manifest["splits"].items()})
print("Durable prepared inputs:", PREPARED_DRIVE)


In [ ]:
# This loads the original model without adding <SA> or the anchor wrapper.
# Downloaded weights live in the Drive Hugging Face cache and are reused later.
run_project("scripts/smoke_pretrained_dsa.py", "--backbone", BACKBONE,
            "--revision", MODEL_REVISION, "--device", DEVICE, "--output", SMOKE_REPORT)
smoke = json.loads(SMOKE_REPORT.read_text())
assert smoke["passed"] and not smoke["vocabulary_changed"] and not smoke["anchor_wrapper_used"]
print("Unmodified checkpoint smoke passed; this is an inference check, not an accuracy result.")


Training starts from the same seed and data order for each policy. FP32 and
activation checkpointing keep this first pilot simple. Updates and loss terms
are logged; complete model/optimizer/RNG checkpoints and a latest pointer are
saved on Drive. Two complete generations are kept per arm. Reserve several GB
per arm, and inspect measured memory/speed before scaling up.

After a disconnect, set `REPO_REF` to the recorded `REPO_COMMIT`, then rerun the
preceding cells with the same settings and runtime, followed by this cell. `--resume auto` continues from the latest complete
checkpoint. You can increase `STEPS` to extend the run. Changed hardware,
dependencies, implementation or experimental settings require a fresh run name;
exact continuation across different GPUs is not claimed.


In [ ]:
training_reports = {}
for arm in ARMS:
    print(f"Training/resuming {arm}: target {STEPS} updates", flush=True)
    run_dir = RUN_ROOT / arm
    run_project("scripts/train_dsa.py", "--prepared-dir", PREPARED_LOCAL,
                "--output-dir", run_dir, "--backbone", BACKBONE, "--revision", MODEL_REVISION,
                "--device", DEVICE, "--write-mode", arm, "--loops", LOOPS, "--workspace", WORKSPACE,
                "--steps", STEPS, "--learning-rate", LEARNING_RATE, "--seed", SEED,
                "--gradient-accumulation", GRADIENT_ACCUMULATION,
                "--save-every", SAVE_EVERY, "--keep-checkpoints", KEEP_CHECKPOINTS,
                "--resume", "auto", "--smoke-report", SMOKE_REPORT)
    training_reports[arm] = json.loads((run_dir / "train_report.json").read_text())
print("Training reports saved:", RUN_ROOT)


Validation generation receives question text only. Gold answers are used afterward
for scoring. The default 32-question result is a pilot diagnostic; set
`VALID_LIMIT=0` for the complete 500-question validation split. The evaluator saves
each prediction and can resume interrupted evaluations. Output names include the
checkpoint step, loop budget, validation limit and generation cap.


In [ ]:
def checkpoint_step(arm):
    return json.loads((RUN_ROOT / arm / "latest.json").read_text())["step"]

def evaluation_dir(arm, split, budget, limit):
    return REPORT_ROOT / arm / f"{split}-step{checkpoint_step(arm)}-k{budget}-n{limit}-g{MAX_NEW_TOKENS}"

validation_reports = []
for arm in ARMS:
    for budget in EVAL_BUDGETS:
        out = evaluation_dir(arm, "valid", budget, VALID_LIMIT)
        run_project("scripts/evaluate_dsa.py", "--prepared-dir", PREPARED_LOCAL,
                    "--run-dir", RUN_ROOT / arm, "--output-dir", out,
                    "--split", "valid", "--loop-budget", budget, "--limit", VALID_LIMIT,
                    "--max-new-tokens", MAX_NEW_TOKENS, "--device", DEVICE)
        validation_reports.append({"arm": arm, **json.loads((out / "report.json").read_text())})
print("Validation predictions and reports saved:", REPORT_ROOT)


## Final held-out evaluation — leave disabled during development

Freeze the training setup, checkpoint choice, decoding settings and loop budgets
using validation before enabling this cell. It evaluates **every one of the 1,319
test questions** for every declared arm/budget with the pinned LOTUS scorer. Test
answers do not select checkpoints or tune any setting. A small-subset training
run still remains a pilot even if it evaluates the full test split.


In [ ]:
RUN_TEST = False  # Enable only after freezing the complete evaluation protocol.
final_reports = []
if RUN_TEST:
    for arm in ARMS:
        for budget in EVAL_BUDGETS:
            out = evaluation_dir(arm, "test", budget, 0)
            run_project("scripts/evaluate_dsa.py", "--prepared-dir", PREPARED_LOCAL,
                        "--run-dir", RUN_ROOT / arm, "--output-dir", out,
                        "--split", "test", "--loop-budget", budget, "--limit", 0,
                        "--max-new-tokens", MAX_NEW_TOKENS, "--device", DEVICE, "--final-test")
            result = json.loads((out / "report.json").read_text())
            assert result["total"] == 1319 and result["all_source_examples_retained"]
            final_reports.append({"arm": arm, **result})
else:
    print("Held-out test evaluation is disabled. Use validation for pilot decisions.")


In [ ]:
print(f"{'arm':12} {'split':6} {'K':>3} {'n':>5} {'accuracy':>9} {'invalid':>9} {'sec/q':>9}")
for report in validation_reports + final_reports:
    print(f"{report['arm']:12} {report['split']:6} {report['loop_budget']:3d} {report['total']:5d} "
          f"{report['accuracy']:9.3%} {report['invalid_answer_rate']:9.3%} "
          f"{report['average_latency_seconds']:9.3f}")
training_profiles = {}
for arm, report in training_reports.items():
    with (RUN_ROOT / arm / "metrics.jsonl").open() as journal:
        metrics = [json.loads(line) for line in journal if line.strip()]
    measured_seconds = sum(row["seconds"] for row in metrics)
    profile = {"completed_steps": report["step"], "measured_update_seconds": measured_seconds,
               "updates_per_second": len(metrics) / measured_seconds if measured_seconds else None,
               "peak_process_rss_bytes": max((row["peak_process_rss_bytes"] for row in metrics), default=0),
               "peak_cuda_allocated_bytes": max((row["peak_cuda_allocated_bytes"] or 0 for row in metrics), default=0),
               "first_total_loss": metrics[0]["total"] if metrics else None,
               "last_total_loss": metrics[-1]["total"] if metrics else None,
               "mean_write_rate": sum(row["write_rate"] for row in metrics) / len(metrics) if metrics else None}
    training_profiles[arm] = profile
    print(arm, "training profile:", json.dumps(profile, indent=2))
summary = {"config": config, "training_profiles": training_profiles, "prepared_manifest_sha256": hashlib.sha256((PREPARED_LOCAL / 'manifest.json').read_bytes()).hexdigest(),
           "training": training_reports, "validation": validation_reports, "final_test": final_reports,
           "purpose": "implementation pilot; no comparative research claim"}
summary_path = REPORT_ROOT / "notebook-summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
print("Persistent run summary:", summary_path)
print("Reconnect: run setup, reuse cached data/model files, then rerun training to resume.")
